<a href="https://colab.research.google.com/github/Kishanmvs/Data-Mining-Project/blob/main/Module_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Pattern Mining (Apriori & FP-Growth)
## BCI606 - Data Mining Mini Project
### Domain: Global Space Exploration Analytics

---

**Objective:** Discover association rules in space mission data using Apriori and FP-Growth algorithms.

Since the dataset mixes numerical and categorical features, we first **discretize** continuous features into meaningful categories (e.g., , ), then mine for patterns like:

>  ⟹  — Support: 5.2%, Confidence: 68%, Lift: 2.1x


## 📦 Step 0: Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import time
import tracemalloc
import warnings
import os

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

warnings.filterwarnings("ignore")
os.makedirs("datasets", exist_ok=True)

print("All libraries imported successfully ✓")


All libraries imported successfully ✓


## 📂 Step 1: Load Raw Data

> We reload the **original raw data** (not Z-score normalized) because discretization needs original value ranges.


In [ ]:
# Load dataset
df = pd.read_csv("Global_Space_Exploration_Dataset.csv")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


Dataset shape: (3000, 12)
Columns: ['Country', 'Year', 'Mission Name', 'Mission Type', 'Launch Site', 'Satellite Type', 'Budget (in Billion $)', 'Success Rate (%)', 'Technology Used', 'Environmental Impact', 'Collaborating Countries', 'Duration (in Days)']


,Country,Year,Mission Name,Mission Type,Launch Site,Satellite Type,Budget (in Billion $),Success Rate (%),Technology Used,Environmental Impact,Collaborating Countries,Duration (in Days)
0,China,2008,Sharable tertiary superstructure,Manned,Sheilatown,Communication,16.20,90,Nuclear Propulsion,Medium,"France, UK, Russia",112
1,Japan,2018,Re-engineered composite flexibility,Manned,New Ericfurt,Communication,29.04,99,Solar Propulsion,High,"Germany, Israel",236
2,Israel,2013,Reactive disintermediate projection,Manned,Port Kaitlynstad,Communication,28.73,54,AI Navigation,Medium,"China, Israel, USA",238
3,UAE,2010,Grass-roots 6thgeneration implementation,Unmanned,Mariastad,Spy,37.27,58,Traditional Rocket,Low,USA,186
4,India,2006,Balanced discrete orchestration,Manned,North Jasonborough,Weather,18.95,91,Solar Propulsion,Medium,"Israel, China, India",277


### Feature Engineering

In [ ]:
# Drop identifiers and raw target
df.drop(columns=["Mission Name", "Launch Site"], inplace=True)

# Engineer collaboration count
df["collab_count"] = df["Collaborating Countries"].apply(
    lambda x: len([c.strip() for c in str(x).split(",") if c.strip()])
)
df.drop(columns=["Collaborating Countries"], inplace=True)

# Binarize target outcome
THRESHOLD = 70
df["mission_outcome"] = (df["Success Rate (%)"] >= THRESHOLD).map(
    {True: "HIGH_SUCCESS", False: "AT_RISK"}
)
df.drop(columns=["Success Rate (%)"], inplace=True)

print(f"Records: {len(df):,}   Columns after engineering: {df.shape[1]}")
print(f"HIGH_SUCCESS missions : {(df["mission_outcome"]=="HIGH_SUCCESS").sum():,}")
print(f"AT_RISK missions       : {(df["mission_outcome"]=="AT_RISK").sum():,}")
df.head(3)


Records: 3,000   Columns after engineering: 10
HIGH_SUCCESS missions : 1,802
AT_RISK missions       : 1,198


,Country,Year,Mission Type,Satellite Type,Budget (in Billion $),Technology Used,Environmental Impact,Duration (in Days),collab_count,mission_outcome
0,China,2008,Manned,Communication,16.20,Nuclear Propulsion,Medium,112,3,HIGH_SUCCESS
1,Japan,2018,Manned,Communication,29.04,Solar Propulsion,High,236,2,HIGH_SUCCESS
2,Israel,2013,Manned,Communication,28.73,AI Navigation,Medium,238,3,AT_RISK


## 🪣 Step 2: Discretize Numerical Features into Categorical Bins

| Feature | Bins | Labels |
|---------|------|--------|
| Budget (\) | 0 / 12.5 / 25 / 40 / ∞ | LOW / MED / HIGH / VHIGH |
| Duration (days) | 0 / 91 / 180 / 272 / ∞ | SHORT / MED / LONG / VLONG |
| Collaborators | 0 / 1 / 2 / 3 / ∞ | SOLO / PAIR / TRIO / MANY |
| Year | ≤2009 / ≤2017 / 2018+ | EARLY / MID / MODERN |


In [ ]:
df_disc = pd.DataFrame()

# Budget bins
df_disc["budget"] = pd.cut(df["Budget (in Billion $)"],
    bins=[0, 12.5, 25, 40, float("inf")],
    labels=["budget_LOW", "budget_MED", "budget_HIGH", "budget_VHIGH"],
    include_lowest=True)

# Duration bins
df_disc["duration"] = pd.cut(df["Duration (in Days)"],
    bins=[0, 91, 180, 272, float("inf")],
    labels=["dur_SHORT", "dur_MED", "dur_LONG", "dur_VLONG"],
    include_lowest=True)

# Collaboration count bins
df_disc["collab"] = pd.cut(df["collab_count"],
    bins=[-1, 1, 2, 3, float("inf")],
    labels=["collab_SOLO", "collab_PAIR", "collab_TRIO", "collab_MANY"])

# Year -> Mission Era
def assign_era(year):
    if year <= 2009:  return "era_EARLY"
    elif year <= 2017: return "era_MID"
    else:             return "era_MODERN"

df_disc["era"]          = df["Year"].apply(assign_era)
df_disc["country"]      = "cntry_" + df["Country"].astype(str)
df_disc["mission_type"] = "type_"  + df["Mission Type"].astype(str)
df_disc["satellite"]    = "sat_"   + df["Satellite Type"].astype(str)
df_disc["technology"]   = "tech_"  + df["Technology Used"].astype(str)
df_disc["env_impact"]   = "env_"   + df["Environmental Impact"].astype(str)
df_disc["outcome"]      = df["mission_outcome"]

print("Discretized features:")
for col in df_disc.columns:
    print(f"  {col:15s}: {list(df_disc[col].dropna().unique()[:5])}")


Discretized features:
  budget         : ['budget_MED', 'budget_HIGH', 'budget_LOW', 'budget_VHIGH']
  duration       : ['dur_MED', 'dur_LONG', 'dur_VLONG', 'dur_SHORT']
  collab         : ['collab_TRIO', 'collab_PAIR', 'collab_SOLO']
  era            : ['era_EARLY', 'era_MODERN', 'era_MID']
  country        : ['cntry_China', 'cntry_Japan', 'cntry_Israel', 'cntry_UAE', 'cntry_India']
  mission_type   : ['type_Manned', 'type_Unmanned']
  satellite      : ['sat_Communication', 'sat_Spy', 'sat_Weather', 'sat_Research', 'sat_Navigation']
  technology     : ['tech_Nuclear Propulsion', 'tech_Solar Propulsion', 'tech_AI Navigation', 'tech_Traditional Rocket', 'tech_Reusable Rocket']
  env_impact     : ['env_Medium', 'env_High', 'env_Low']
  outcome        : ['HIGH_SUCCESS', 'AT_RISK']


## 🧮 Step 3: One-Hot Encode into Boolean Basket Matrix

- Each **row** = one mission (transaction)
- Each **column** = one item (e.g., )
- Values are  /


In [ ]:
basket = pd.get_dummies(df_disc).astype(bool)

print(f"Basket shape        : {basket.shape}")
print(f"Unique items (cols) : {basket.shape[1]}")
print(f"Total missions      : {basket.shape[0]}")
basket.head(3)


Basket shape        : (3000, 42)
Unique items (cols) : 42
Total missions      : 3000


,budget_budget_LOW,budget_budget_MED,budget_budget_HIGH,budget_budget_VHIGH,duration_dur_SHORT,duration_dur_MED,duration_dur_LONG,duration_dur_VLONG,collab_collab_SOLO,collab_collab_PAIR,...,technology_tech_AI Navigation,technology_tech_Nuclear Propulsion,technology_tech_Reusable Rocket,technology_tech_Solar Propulsion,technology_tech_Traditional Rocket,env_impact_env_High,env_impact_env_Low,env_impact_env_Medium,outcome_AT_RISK,outcome_HIGH_SUCCESS
0,False,True,False,False,False,True,False,False,False,False,...,False,True,False,False,False,False,False,True,False,True
1,False,False,True,False,False,False,True,False,False,True,...,False,False,False,True,False,True,False,False,False,True
2,False,False,True,False,False,False,True,False,False,False,...,True,False,False,False,False,False,False,True,True,False


## ⚙️ Step 4: Apriori Algorithm

**How it works:**
1. Scan DB → find all frequent 1-itemsets (support ≥ min_support)
2. Generate candidate k+1-itemsets from frequent k-itemsets
3. Scan DB again to count support; prune infrequent
4. Repeat until no new frequent itemsets found

**Anti-Monotone property:** *If an itemset is infrequent, all its supersets are too* — enables aggressive pruning.


In [ ]:
tracemalloc.start()
t_start = time.time()

freq_apriori = apriori(basket, min_support=0.05, use_colnames=True, max_len=3, verbose=0)

t_apriori       = time.time() - t_start
mem_apriori_peak = tracemalloc.get_traced_memory()[1] / (1024 * 1024)
tracemalloc.stop()

print(f"Time        : {t_apriori:.2f} s")
print(f"Peak memory : {mem_apriori_peak:.2f} MB")
print(f"Frequent itemsets found : {len(freq_apriori)}")
if len(freq_apriori) > 0:
    print(f"Max itemset length : {freq_apriori["itemsets"].apply(len).max()}")
freq_apriori.sort_values("support", ascending=False).head(10)


Time        : 0.13 s
Peak memory : 49.91 MB
Frequent itemsets found : 619
Max itemset length : 3


,support,itemsets
40,0.600667,(outcome_HIGH_SUCCESS)
24,0.509333,(mission_type_type_Manned)
25,0.490667,(mission_type_type_Unmanned)
39,0.399333,(outcome_AT_RISK)
11,0.379000,(era_era_EARLY)
8,0.346667,(collab_collab_SOLO)
38,0.344000,(env_impact_env_Medium)
37,0.336667,(env_impact_env_Low)
10,0.328667,(collab_collab_TRIO)
9,0.324667,(collab_collab_PAIR)


## 🌳 Step 5: FP-Growth Algorithm

**How it works:**
1. Scan DB **once** → find frequent 1-itemsets
2. Scan DB **once more** → build a compressed FP-Tree (prefix tree)
3. Mine frequent patterns from the FP-Tree via conditional pattern bases — **no explicit candidate generation**

**Advantage:** Only **2 database scans** vs. *k* scans in Apriori → typically 5–10× faster on dense data.


In [ ]:
tracemalloc.start()
t_start = time.time()

freq_fpgrowth = fpgrowth(basket, min_support=0.05, use_colnames=True, max_len=3, verbose=0)

t_fpgrowth       = time.time() - t_start
mem_fpgrowth_peak = tracemalloc.get_traced_memory()[1] / (1024 * 1024)
tracemalloc.stop()

speedup = t_apriori / t_fpgrowth if t_fpgrowth > 0 else float("inf")

print(f"Time        : {t_fpgrowth:.2f} s")
print(f"Peak memory : {mem_fpgrowth_peak:.2f} MB")
print(f"Frequent itemsets found : {len(freq_fpgrowth)}")
print(f"FP-Growth is {speedup:.1f}x faster than Apriori")
freq_fpgrowth.sort_values("support", ascending=False).head(10)


Time        : 79.44 s
Peak memory : 5.87 MB
Frequent itemsets found : 619
FP-Growth is 0.0x faster than Apriori


,support,itemsets
0,0.600667,(outcome_HIGH_SUCCESS)
1,0.509333,(mission_type_type_Manned)
21,0.490667,(mission_type_type_Unmanned)
17,0.399333,(outcome_AT_RISK)
2,0.379000,(era_era_EARLY)
22,0.346667,(collab_collab_SOLO)
3,0.344000,(env_impact_env_Medium)
23,0.336667,(env_impact_env_Low)
4,0.328667,(collab_collab_TRIO)
10,0.324667,(collab_collab_PAIR)


### 📊 Algorithm Comparison Table

In [ ]:
comparison = {
    "Metric": ["Execution Time (s)", "Peak Memory (MB)", "Frequent Itemsets Found",
               "DB Scans Required", "Candidate Generation?", "Data Structure", "Best For"],
    "Apriori": [f"{t_apriori:.2f}", f"{mem_apriori_peak:.2f}", str(len(freq_apriori)),
                "k per itemset level", "Yes (generates all k+1 candidates)",
                "Hash tree", "Small datasets, easy to implement"],
    "FP-Growth": [f"{t_fpgrowth:.2f}", f"{mem_fpgrowth_peak:.2f}", str(len(freq_fpgrowth)),
                  "2 (always)", "No (tree-based mining)",
                  "Prefix FP-Tree (compressed)", "Large datasets, memory-efficient"],
}
pd.DataFrame(comparison)


,Metric,Apriori,FP-Growth
0,Execution Time (s),0.13,79.44
1,Peak Memory (MB),49.91,5.87
2,Frequent Itemsets Found,619,619
3,DB Scans Required,k per itemset level,2 (always)
4,Candidate Generation?,Yes (generates all k+1 candidates),No (tree-based mining)
5,Data Structure,Hash tree,Prefix FP-Tree (compressed)
6,Best For,"Small datasets, easy to implement","Large datasets, memory-efficient"


## 📐 Step 6: Generate Association Rules

| Metric | Formula | Meaning |
|--------|---------|--------|
| **Support** | P(A ∩ B) | How common is this combination? |
| **Confidence** | P(B \| A) | Given A, how likely is B? |
| **Lift** | Conf(A→B) / P(B) | How much does A promote B vs. random? Lift > 1 = positive association |


In [ ]:
rules = association_rules(freq_fpgrowth, metric="lift", min_threshold=1.2,
                          num_itemsets=len(basket))

print(f"Total rules generated: {len(rules)}")

# Filter by consequent
success_rules = rules[rules["consequents"].apply(lambda x: "outcome_HIGH_SUCCESS" in x)]
risk_rules    = rules[rules["consequents"].apply(lambda x: "outcome_AT_RISK" in x)]

success_rules = success_rules.sort_values("lift", ascending=False)
risk_rules    = risk_rules.sort_values("lift", ascending=False)

print(f"Rules predicting HIGH_SUCCESS : {len(success_rules)}")
print(f"Rules predicting AT_RISK      : {len(risk_rules)}")


Total rules generated: 8
Rules predicting HIGH_SUCCESS : 2
Rules predicting AT_RISK      : 0


### 🟢 Top 10 HIGH_SUCCESS Predicting Rules

In [ ]:
disp = success_rules.head(10).copy()
disp["antecedents"] = disp["antecedents"].apply(lambda x: ", ".join(sorted(x)))
disp["consequents"] = disp["consequents"].apply(lambda x: ", ".join(sorted(x)))
disp[["antecedents", "consequents", "support", "confidence", "lift"]].reset_index(drop=True)


,antecedents,consequents,support,confidence,lift
0,env_impact_env_Low,"duration_dur_SHORT, outcome_HIGH_SUCCESS",0.061333,0.182178,1.244954
1,satellite_sat_Research,"collab_collab_PAIR, outcome_HIGH_SUCCESS",0.052000,0.240370,1.222219


### 🔴 Top 10 AT_RISK Predicting Rules

In [ ]:
disp2 = risk_rules.head(10).copy()
disp2["antecedents"] = disp2["antecedents"].apply(lambda x: ", ".join(sorted(x)))
disp2["consequents"] = disp2["consequents"].apply(lambda x: ", ".join(sorted(x)))
disp2[["antecedents", "consequents", "support", "confidence", "lift"]].reset_index(drop=True)


,antecedents,consequents,support,confidence,lift


## 📦 Step 7: Top Frequent Itemsets (by Support)

In [ ]:
freq_sorted = freq_fpgrowth.sort_values("support", ascending=False)
multi_item  = freq_sorted[freq_sorted["itemsets"].apply(len) >= 2].head(15).copy()
multi_item["itemsets"] = multi_item["itemsets"].apply(lambda x: ", ".join(sorted(x)))
multi_item.reset_index(drop=True)


,support,itemsets
0,0.312000,"mission_type_type_Manned, outcome_HIGH_SUCCESS"
1,0.288667,"mission_type_type_Unmanned, outcome_HIGH_SUCCESS"
2,0.227333,"era_era_EARLY, outcome_HIGH_SUCCESS"
3,0.207667,"env_impact_env_Medium, outcome_HIGH_SUCCESS"
4,0.205333,"collab_collab_SOLO, outcome_HIGH_SUCCESS"
5,0.204000,"env_impact_env_Low, outcome_HIGH_SUCCESS"
6,0.202000,"mission_type_type_Unmanned, outcome_AT_RISK"
7,0.198667,"collab_collab_TRIO, outcome_HIGH_SUCCESS"
8,0.197333,"mission_type_type_Manned, outcome_AT_RISK"
9,0.196667,"collab_collab_PAIR, outcome_HIGH_SUCCESS"


## 💾 Step 8: Save Results to CSV

In [ ]:
def save_rules(df_rules, path):
    out = df_rules.copy()
    out["antecedents"] = out["antecedents"].apply(lambda x: ", ".join(sorted(x)))
    out["consequents"] = out["consequents"].apply(lambda x: ", ".join(sorted(x)))
    out.to_csv(path, index=False)
    print(f"Saved: {path}  ({len(out)} rows)")

save_rules(success_rules, "datasets/success_association_rules.csv")
save_rules(risk_rules,    "datasets/risk_association_rules.csv")

freq_out = freq_fpgrowth.copy()
freq_out["itemsets"] = freq_out["itemsets"].apply(lambda x: ", ".join(sorted(x)))
freq_out.to_csv("datasets/frequent_itemsets.csv", index=False)
print("Saved: datasets/frequent_itemsets.csv")

print("
✅ Module 3 Complete!")
